In [ ]:
code = 'DT'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output\\'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def DT(bt, start_time, end_time, orderside, method, decay, sl, om):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None

        from_candle_close = True if method == 'CC' else False
        entry_time = start_dt

        # check entry
        ce_open, ce_high, ce_low, ce_close, ce_decay_price, ce_decay_flag, ce_decay_time = bt.decay_check_single_leg(start_dt, end_dt, ce_scrip, decay=decay, from_candle_close=from_candle_close, orderside=orderside, with_ohlc=True)
        pe_open, pe_high, pe_low, pe_close, pe_decay_price, pe_decay_flag, pe_decay_time = bt.decay_check_single_leg(start_dt, end_dt, pe_scrip, decay=decay, from_candle_close=from_candle_close, orderside=orderside, with_ohlc=True)
        
        # check sl
        ce_sl_price, ce_sl_flag, _, _, ce_sl_time, ce_pnl = bt.sl_check_single_leg(ce_decay_time, end_dt, ce_scrip, o=(None if method == 'CC' else ce_decay_price), sl=sl, orderside=orderside, from_candle_close=from_candle_close)
        pe_sl_price, pe_sl_flag, _, _, pe_sl_time, pe_pnl = bt.sl_check_single_leg(pe_decay_time, end_dt, pe_scrip, o=(None if method == 'CC' else pe_decay_price), sl=sl, orderside=orderside, from_candle_close=from_candle_close)

        total_pnl = ce_pnl + pe_pnl
        return [code, bt.index, start_time, end_time, orderside, method, decay, sl, om, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price, ce_scrip, ce_open, ce_high, ce_low, ce_close, ce_decay_price, ce_decay_flag, ce_decay_time, ce_sl_price, ce_sl_flag, ce_sl_time, pe_scrip, pe_open, pe_high, pe_low, pe_close, pe_decay_price, pe_decay_flag, pe_decay_time, pe_sl_price, pe_sl_flag, pe_sl_time, ce_pnl, pe_pnl, total_pnl]
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, method, decay, sl, om])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)
            
            log_cols = ('P_Strategy/P_Index/P_StartTime/P_EndTime/P_OrderSide/P_Method/P_Decay/P_SL/P_OM/Date/Day/DTE/EntryTime/Future/CE.Strike/CE.Open/CE.High/CE.Low/CE.Close/CE.Decay/CE.Decay.Flag/CE.Decay.Time/CE.SL.Price/CE.SL.Flag/CE.SL.Time/PE.Strike/PE.Open/PE.High/PE.Low/PE.Close/PE.Decay/PE.Decay.Flag/PE.Decay.Time/PE.SL.Price/PE.SL.Flag/PE.SL.Time/CE.PNL/PE.PNL/Total.PNL').split('/')

            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)
                        
                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [DT(bt, row['entry_time'], row['exit_time'], row['orderside'], row['method'], row['decay'], row['sl'], row['om']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)

                    del bt
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))